# Drishti — End-to-End App Test (on Colab)

Runs the **real `app/` code** — router, engines, guardrail, translation, speech — against
real photos, on a Colab T4. This is the Phase-1 exit criterion, executed without installing
anything locally.

Everything in `app/` is unit-tested with fakes (124 tests). What has never happened is a
real model loading through it. That is what this notebook checks.

### Why the repo has to be fetched

The Colab VS Code extension runs *cells* on a Colab machine; it does not copy your project
there. `app/` therefore does not exist on the runtime until §1 fetches it — via `git clone`
(recommended) or a zip upload.

### Two phases, one mandatory restart — read this before running

PaddlePaddle (OCR) and PyTorch (translation, speech, the VLM) each bundle their own OpenMP
runtime. Co-loading them **crashes the kernel** — confirmed during development, not just a
theoretical risk: running OCR then IndicTransToolkit in one session triggered
`AsyncIOLoopKernelRestarter: restarting kernel` with no Python traceback (`DEC-006`).

So the notebook is split:

- **§4–5 (PaddlePaddle only):** medicine mode, Devanagari read mode. Result saved to a JSON
  checkpoint on disk.
- **§6–7 (PyTorch only):** translation, speech, the VLM. Reads the checkpoint instead of a
  Python variable, because the restart between phases clears all in-memory state.

**You must restart the runtime between §5 and §6** — the notebook tells you exactly when,
and the bootstrap cell after the restart needs no re-clone or re-install.

**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
import platform, os
print(platform.system(), platform.release())
print('hostname:', platform.node())
print('cwd:', os.getcwd())
!nvidia-smi --query-gpu=name --format=csv,noheader


## 1. Get the project onto the runtime

The Colab VS Code extension runs *cells* on Colab hardware but leaves your files on the
local disk, so `app/` does not exist on the runtime until we put it there.

> **`google.colab.files.upload()` does not work from VS Code.** It is a browser widget: the
> HTML renders, the JavaScript bridge that picks the file never loads, and the cell hangs
> until you interrupt it. Use one of the two paths below instead.

### Path A — git clone (recommended)

Push the project to GitHub once, then set `REPO_URL` below. Every later run is a single cell
that always pulls current code, and the repo ends up backed up and shareable with your guide.

```powershell
git remote add origin https://github.com/<you>/drishti.git
git push -u origin main
```

### Path B — run this notebook in the Colab browser

Open [colab.research.google.com](https://colab.research.google.com), upload this notebook,
and `files.upload()` behaves normally. Zero setup, but you lose the VS Code editor.

Sample photos are committed under `data/samples/`, so **no image upload is needed either
way** — that failure mode is gone entirely.

In [ ]:
import os

# Set before any framework import -- see DEC-006.
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

import shutil, subprocess, sys, zipfile
from pathlib import Path

REPO_URL = 'https://github.com/DevGurav/Drishti.git'
REFRESH = True         # re-fetch every run; set False only to keep a hand-edited runtime
PROJECT = Path('/content/drishti')
WORKDIR = Path('/content')

CLONE_HELP = """
If the repository is PRIVATE, clone with a token:
  GitHub -> Settings -> Developer settings -> Personal access tokens
  -> Fine-grained token, Repository access: this repo, Contents: Read
  REPO_URL = 'https://<TOKEN>@github.com/DevGurav/Drishti.git'

Otherwise: make the repo public, or use Path B (Colab browser + zip upload).
"""


def _find_project_root(start: Path):
    """Locate the folder holding app/router.py, however the archive nested it."""
    if (start / 'app' / 'router.py').exists():
        return start
    for marker in start.glob('*/app/router.py'):
        return marker.parents[1]
    return None


# Step out of PROJECT before deleting it. A re-run leaves the process cwd inside the
# project, and deleting the directory you are standing in makes every later subprocess
# fail with "unable to read current working directory" -- git exits 128.
os.chdir(WORKDIR)

# A stale copy is the failure that actually bites: the runtime keeps whatever was fetched
# first, so pushing new code changes nothing here and the error surfaces somewhere else.
if REFRESH:
    for stale in (PROJECT, WORKDIR / '_clone', WORKDIR / '_unpack'):
        shutil.rmtree(stale, ignore_errors=True)

if not (PROJECT / 'app' / 'router.py').exists():
    if REPO_URL:
        clone = subprocess.run(
            ['git', 'clone', '--depth', '1', REPO_URL, str(WORKDIR / '_clone')],
            capture_output=True, text=True)
        if clone.returncode != 0:
            # Print git's own message. check=True hides stderr, and the cause is usually
            # only visible there (private repo, typo, auth).
            print(f'git clone failed (exit {clone.returncode}):')
            print(clone.stderr.strip())
            print(CLONE_HELP)
            raise SystemExit('clone failed -- see the message above')
        found = _find_project_root(WORKDIR / '_clone')
        if found is None:
            raise SystemExit('app/router.py not found in the cloned repo.')
        shutil.move(str(found), str(PROJECT))
    else:
        # Path B only -- this widget works in the Colab browser, never from VS Code.
        try:
            from google.colab import files
        except ImportError:
            raise SystemExit('Not running on Colab. Set REPO_URL above.')
        print('Upload drishti.zip  (Colab browser only; from VS Code set REPO_URL instead)')
        up = files.upload()
        with zipfile.ZipFile(next(iter(up))) as z:
            z.extractall(WORKDIR / '_unpack')
        found = _find_project_root(WORKDIR / '_unpack')
        if found is None:
            raise SystemExit('app/router.py not in the archive -- did you zip the drishti '
                             'folder itself?')
        shutil.move(str(found), str(PROJECT))

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

samples = sorted((PROJECT / 'data' / 'samples').glob('*.jpg'))
head = subprocess.run(['git', '-C', str(PROJECT), 'log', '--oneline', '-1'],
                      capture_output=True, text=True).stdout.strip()

print('project root :', PROJECT)
print('fetched HEAD :', head or '(not a git checkout)')
print('modes        :', sorted(p.stem for p in (PROJECT / 'app' / 'modes').glob('[!_]*.py')))
print('sample images:', [p.name for p in samples] or 'NONE')

# Fail here, naming the real cause, rather than three cells later with a missing file.
if not samples:
    print('data/samples/ is empty, so the fetched code is out of date.')
    print('Most likely your local commits have not been pushed yet:')
    print('    git push')
    print('then re-run this cell (REFRESH=True forces a fresh clone).')
    raise SystemExit('stale code -- see the message above')

In [ ]:
# The suite needs no models, so a pass here proves the upload is complete and importable
# before we spend minutes downloading weights.
!python -m unittest discover -s tests -t . 2>&1 | tail -4

## 2. Install engines

Weights are **not** downloaded here — every engine loads lazily on first use, so each mode
below pays only for what it needs.

> **Ignore this warning if you see it:** `gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0,
> but you have huggingface-hub 0.36.2 which is incompatible.` Colab's base image ships
> `gradio` pre-installed; installing `transformers`/`IndicTransToolkit` pins an older
> `huggingface-hub` than gradio declares it wants. Nothing in this project imports `gradio`,
> so the conflict is real but inert.

In [ ]:
# transformers<5 is required here specifically. IndicTransToolkit imports
# PreTrainedTokenizerBase from transformers.tokenization_utils, which v5 removed --
# confirmed by the ImportError this notebook hit. SmolVLM (Section 7) is NOT the reason
# for the pin: it already runs fine on transformers 4.x, proven in notebook 00's spike
# (transformers 4.57.6, SmolVLM answered correctly). Pinning down satisfies both engines
# in one process instead of needing them in separate ones.
%pip install -q "transformers<5" paddlepaddle paddleocr IndicTransToolkit

# transformers was already imported once in this kernel (the failed attempt above), so a
# pip downgrade alone has no effect until the cached module is purged -- same fix as the
# surya-ocr version issue in notebook 00b.
import sys
for _mod in [m for m in sys.modules if m == 'transformers' or m.startswith('transformers.')]:
    del sys.modules[_mod]

import transformers
print('transformers:', transformers.__version__)
print('installed')

## 3. Choose test photos

Committed fixtures in `data/samples/` are used by default, so nothing needs uploading.

`strip_paracip.jpg` is the read that produced 55 OCR lines including the drug name,
`EXP.OCT.2026` and `Rs.10.30`. `strip_partial.jpg` is the same strip framed badly — only
3 lines — kept as the negative case.

To test **Devanagari Read mode**, drop a photo containing Marathi or Hindi text into
`data/samples/` and set `DEVANAGARI` below. That is still the one unverified claim in the
project: the code path is confirmed, the model has never seen actual Devanagari.

In [ ]:
SAMPLES = PROJECT / 'data' / 'samples'
photos = sorted(SAMPLES.glob('*.jpg'))

for i, p in enumerate(photos):
    print(f'  [{i}] {p.name}  ({p.stat().st_size/1e3:.0f} KB)')

STRIP = SAMPLES / 'strip_paracip.jpg'      # the good read
DEVANAGARI = None                          # <-- set to a Marathi/Hindi photo to test §6

if not STRIP.exists():
    raise SystemExit(f'{STRIP} missing -- is data/samples/ present in the project?')

print('\nstrip     :', STRIP.name)
print('devanagari:', DEVANAGARI.name if DEVANAGARI else '(none set -- §6 will skip)')

## 4. Medicine mode — the guardrail, end to end

OCR reads the strip, the drug name is matched against the verified database, expiry and MRP
are parsed. If OCR cannot produce a verified name the mode **declines** rather than guessing
(`DEC-007`).

In [ ]:
import time

from app.drug_db import DrugDatabase
from app.engines.paddle_ocr import PaddleOCREngine
from app.modes.medicine import run as run_medicine

ocr = PaddleOCREngine(lang='en')

t0 = time.time()
result = run_medicine(STRIP, ocr, DrugDatabase.from_file())
elapsed = time.time() - t0

print(f'--- medicine mode  ({elapsed:.1f}s) ---')
print('verified :', result.ok)
print('drug     :', result.drug_name)
print('expiry   :', result.expiry_raw, '| expired:', result.expired)
print('MRP      :', result.mrp)
print('\nSPOKEN   :', result.message_en)

if not result.ok:
    print('\nDeclined. Either OCR missed the name, or it is absent from')
    print('data/drug_names_seed.txt -- a 30-entry placeholder, not a real drug database.')

# Checkpointed to disk because the runtime restarts before the torch-based cells run --
# see the markdown cell below for why. A restart wipes every Python variable but leaves
# /content untouched, so this is how 'result' survives into the next phase.
import json

CHECKPOINT = Path('/content/drishti_checkpoint.json')
CHECKPOINT.write_text(json.dumps({
    'ok': result.ok,
    'drug_name': result.drug_name,
    'expiry_raw': result.expiry_raw,
    'expired': result.expired,
    'mrp': result.mrp,
    'message_en': result.message_en,
}))
print(f'\ncheckpoint written: {CHECKPOINT}')

## 5. Read mode — Devanagari

Moved up next to medicine mode on purpose: both use **PaddleOCR only**, so they belong in
the same PaddlePaddle-only phase (see the restart notice below).

`lang='mr'` resolves to `devanagari_PP-OCRv5_mobile_rec`. The code path is confirmed; what
has never been tested is the model against actual Devanagari text.

In [ ]:
import json

from app.modes.read import run as run_read

devanagari_text = None
if DEVANAGARI is None:
    print('No Devanagari photo set in §3 -- skipping. Read mode in Marathi stays unverified.')
else:
    t0 = time.time()
    devanagari_text = run_read(DEVANAGARI, PaddleOCREngine(lang='mr'))
    print(f'--- read mode, devanagari ({time.time()-t0:.1f}s) ---')
    print(devanagari_text)

# append to the same checkpoint medicine mode wrote above
data = json.loads(CHECKPOINT.read_text())
data['devanagari_text'] = devanagari_text
CHECKPOINT.write_text(json.dumps(data))
print(f'\ncheckpoint updated: {CHECKPOINT}')

## ⚠️ Restart the runtime now — required, not optional

**Empirically confirmed, not just theoretical:** running PaddleOCR and then IndicTransToolkit
(translation, which loads PyTorch) in the same kernel restarted the kernel mid-run —
`AsyncIOLoopKernelRestarter: restarting kernel`, no Python traceback. This is the same
PaddlePaddle/PyTorch OpenMP collision documented for OCR + the VLM (`DEC-006`), now confirmed
to also break OCR + IndicTransToolkit. `KMP_DUPLICATE_LIB_OK` (set in §1) reduces how often
this happens; it does not eliminate it.

Every cell above this point uses **PaddlePaddle only**. Every cell below uses **PyTorch
only**. That grouping is deliberate so a single restart is enough.

**Do this now:**

1. `Runtime → Restart session` (not "Disconnect and delete runtime" — that would lose the
   installed packages too, which the restart does not need to touch)
2. Run the bootstrap cell immediately below
3. Continue from §6 (Marathi/Hindi speech)

The bootstrap cell does **not** re-clone or re-install anything — a session restart kills
the Python process but leaves the VM's disk, including installed packages, untouched. It
only restores `sys.path`/`cwd` and reloads the checkpoint §4–5 wrote to disk.

In [ ]:
# Bootstrap after the restart above. No git clone, no pip install -- the VM's disk
# (installed packages, the cloned repo) survives a session restart; only Python's
# in-memory state does not.
import json
import os
import sys
import time
from pathlib import Path

os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

PROJECT = Path('/content/drishti')
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

CHECKPOINT = Path('/content/drishti_checkpoint.json')
if not CHECKPOINT.exists():
    raise SystemExit(
        f'{CHECKPOINT} not found. Did the medicine-mode cell (§4) run and finish before '
        'the restart? Scroll up and re-run from §4.'
    )

checkpoint = json.loads(CHECKPOINT.read_text())
print('project root :', PROJECT)
print('checkpoint   :', checkpoint)

## 6. Marathi output and speech — the Phase-1 exit criterion

Runs in the fresh, PyTorch-only session started above. Reads `message_en` from the
checkpoint rather than the `result` variable, which no longer exists after the restart.

In [ ]:
from IPython.display import Audio, display

from app.engines.indictrans import IndicTrans2Translator
from app.engines.mms_tts import MMSTTSEngine
from app.speech import deliver

if not checkpoint.get('message_en'):
    raise SystemExit('checkpoint has no message_en -- did medicine mode (§4) succeed?')

translator = IndicTrans2Translator()
tts = MMSTTSEngine(out_dir=Path('/content/audio'))

for lang in ('mr', 'hi'):
    t0 = time.time()
    spoken = deliver(checkpoint['message_en'], lang=lang, translator=translator,
                     tts=tts, speak=True)
    print(f'--- {lang} ({time.time()-t0:.1f}s) ---')
    print(spoken.text_out)
    display(Audio(str(spoken.audio_path)))

## 7. Scene mode — the VLM

Same PyTorch-only session as §6 — no second restart needed, SmolVLM and IndicTransToolkit
coexist fine since both are PyTorch.

In [ ]:
from app.engines.smolvlm import SmolVLMEngine
from app.modes.ask import run as run_ask
from app.modes.scene import run as run_scene

vlm = SmolVLMEngine()

t0 = time.time()
print('--- scene mode ---')
print(run_scene(STRIP, vlm), f'({time.time()-t0:.1f}s)')

t0 = time.time()
print('\n--- ask mode ---')
print(run_ask(STRIP, vlm, 'what is written on this?'), f'({time.time()-t0:.1f}s)')

## 8. Findings — fill in, then update `docs/BUILD_PLAN.md`

| Check | Result | Latency |
|---|---|---|
| Medicine: drug name verified | | s |
| Medicine: expiry parsed (earliest of multiple dates) | | |
| Medicine: MRP parsed | | |
| Devanagari Read mode | | s |
| Runtime restart completed cleanly, checkpoint reloaded | | |
| Marathi translation readable | | s |
| Marathi speech intelligible | | s |
| Hindi speech intelligible | | s |
| Scene mode | | s |

**Phase 1 is complete when** the medicine row is verified and Marathi audio plays. Tick
those boxes in the build plan and record the latencies against the <8 s target (RISK-1) —
note that these numbers include the one-time PaddleOCR/PyTorch model download, so a repeat
run will be faster.

Ask a Marathi speaker whether the translation and the synthesized voice are actually
understandable — accuracy metrics do not capture intelligibility, and this is the first time
a human can judge the output.

**On the notebook design:** this run is why the notebook is now split into a PaddlePaddle
phase (§4–5) and a PyTorch phase (§6–7) with a mandatory restart between them, bridged by a
JSON checkpoint on disk. That split is not caution for its own sake — it is what fixed an
actual kernel crash reproduced during development (`DEC-006`, extended to cover
IndicTransToolkit as well as the VLM).